# 🔍 Notebook 3: LLM, Intent & Orchestrator Test

**Objective**: Test reasoning layer — intent classification, SQL/Python generation, full query flow.

**Prerequisites**:
1. `.env` configured with working LLM credentials
2. Notebook 2 completed (tables registered in DuckDB)

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────
import sys, os, json, time
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, HTML

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from hybridtablerag.llm.factory import get_llm
from hybridtablerag.storage.store import DuckDBStore
from hybridtablerag.storage.context import ContextStore
from hybridtablerag.reasoning.sql import LLMSQLGenerator
from hybridtablerag.reasoning.intent import IntentClassifier
from hybridtablerag.reasoning.orchestrator import QueryOrchestrator, QueryResult

def log(msg, level="info"):
    colors = {"info": "#2563eb", "warn": "#b45309", "err": "#dc2626", "ok": "#16a34a"}
    display(HTML(f"<div style='font-family:monospace;font-size:.85rem;color:{colors.get(level,'#0f172a')}'>› {msg}</div>"))

In [ ]:
# ── Initialize Components ─────────────────────────────────────────────
log("Initializing LLM and storage…")

# LLM client (reads from .env)
llm = get_llm()
log(f"LLM ready: {type(llm).__name__}", "ok")

# DuckDB connection
store = DuckDBStore(db_path="data/hybridtablerag_test.duckdb")
tables = [t for t in store.list_tables() if t != "chat_history"]

if not tables:
    raise RuntimeError("No tables found in DuckDB. Run Notebook 2 first.")

log(f"Found tables: {tables}", "ok")

# Context store for conversation history
context_store = ContextStore(store.conn)

# SQL generator
sql_generator = LLMSQLGenerator(llm)

# Fake relationships for testing (replace with real ones from normalization)
relationships = []

# Orchestrator
orchestrator = QueryOrchestrator(
    llm=llm,
    store=store,
    context_store=context_store,
    sql_generator=sql_generator,
    table_names=tables,
    relationships=relationships,
    default_table=tables[0],
)
log("Orchestrator ready", "ok")

In [ ]:
# ── Test Intent Classification ────────────────────────────────────────
log("Testing intent classifier…")
classifier = IntentClassifier(llm)

test_queries = [
    ("How many rows are in the main table?", "sql"),
    ("Plot ticket volume by priority", "python"),
    ("Find tickets about login issues", "vector"),
    ("What did I ask earlier?", "conversational"),
]

for query, expected in test_queries:
    intent = classifier.classify(query)
    status = True if intent == expected else False
    log(f"{status} '{query[:40]}...' → {intent} (expected: {expected})", 
        "ok" if intent==expected else "warn")

In [ ]:
# ── Test SQL Generation ───────────────────────────────────────────────
log("Testing SQL generation…")

from hybridtablerag.storage.schema import build_multi_table_schema_context

schema_ctx = build_multi_table_schema_context(store.conn, tables, relationships, bts_log=[])

test_sql_queries = [
    "Count total rows",
    "Show me the top 5 values in the first column",
    "Filter where status is active",
]

for q in test_sql_queries:
    log(f"Generating SQL for: '{q}'")
    try:
        result = sql_generator.generate_sql(
            user_query=q,
            schema_metadata=schema_ctx["tables"],
            relationships=relationships,
            reasoning=True,
        )
        
        if isinstance(result, dict):
            sql = result["sql_query"]
            reasoning = result.get("reasoning", "")
        else:
            sql = result
            reasoning = ""
        
        display(Markdown(f"**Generated SQL**:\n```sql\n{sql}\n```"))
        if reasoning:
            display(Markdown(f"**Reasoning**:\n{reasoning}"))
        
        # Try executing it
        df = store.execute_query(sql)
        display(Markdown(f"**Result**: {len(df)} row(s)"))
        display(df.head() if not df.empty else Markdown("*Empty result*"))
        log("SQL executed successfully", "ok")
        
    except Exception as e:
        log(f"SQL generation/execution failed: {e}", "err")

In [ ]:
# ── Test Full Orchestrator Flow ───────────────────────────────────────
log("Testing full orchestrator.run() flow…")

session_id = "notebook_test_session"

test_scenarios = [
    {
        "query": "How many total records?",
        "expected_intent": "sql",
        "description": "Simple count query"
    },
    {
        "query": "Show me a summary of the data",
        "expected_intent": "python",
        "description": "Should trigger Python analysis"
    },
    # Add vector test if embeddings exist
]

for scenario in test_scenarios:
    log(f"\nScenario: {scenario['description']}")
    log(f"Query: '{scenario['query']}'")
    
    start = time.time()
    result: QueryResult = orchestrator.run(
        user_query=scenario["query"],
        session_id=session_id,
        reasoning=True,
        debug_mode=True,
    )
    elapsed = time.time() - start
    
    # Display result
    display(Markdown(f"### Result"))
    display(Markdown(f"- **Intent**: `{result.intent}` (expected: `{scenario['expected_intent']}`)"))
    display(Markdown(f"- **Success**: {True if result.success else False}"))
    display(Markdown(f"- **Time**: {elapsed:.2f}s"))
    
    if result.sql:
        display(Markdown(f"**Generated SQL**:\n```sql\n{result.sql}\n```"))
    
    if result.dataframe is not None and not result.dataframe.empty:
        display(Markdown(f"**DataFrame result** ({len(result.dataframe)} rows):"))
        display(result.dataframe.head())
    
    if result.python_code:
        display(Markdown(f"**Generated Python**:\n```python\n{result.python_code}\n```"))
    
    if result.chart:
        display(Markdown("**Chart generated**"))
        # In notebook: result.chart.show() if plotly
    
    if result.reasoning:
        display(Markdown(f"**LLM Reasoning**:\n{result.reasoning}"))
    
    if result.bts_log:
        with pd.option_context('display.max_colwidth', None):
            display(Markdown("**BTS Log**:"))
            for entry in result.bts_log:
                display(Markdown(f"- {entry}"))
    
    if not result.success:
        log(f"Error: {result.error}", "err")

In [ ]:
# ── Test Conversation Context Injection ───────────────────────────────
log("Testing conversation history injection…")

# Simulate a follow-up question
follow_up = "Now filter that by the first category"
log(f"Follow-up query: '{follow_up}'")

result = orchestrator.run(
    user_query=follow_up,
    session_id=session_id,
    reasoning=True,
)

display(Markdown(f"**Context used**: {result.context_used[:200]}..." if result.context_used else "No context injected"))

if result.dataframe is not None:
    display(Markdown(f"**Result**: {len(result.dataframe)} row(s)"))
    display(result.dataframe.head())

In [ ]:
# ── Cleanup ───────────────────────────────────────────────────────────
store.close()
log("Connections closed", "ok")

display(Markdown("## Orchestrator Test Complete"))
display(Markdown("""
### Next Steps
1. If all tests pass: Your core pipeline is working!
2. If intent classification is wrong: Tune `reasoning/intent.py` prompts
3. If SQL generation fails: Check schema context enrichment in `storage/schema.py`
4. If Python execution fails: Review sandbox restrictions in `reasoning/python_exec.py`

### Integration
Once notebooks pass, you can confidently:
- Deploy the FastAPI backend
- Re-enable the Streamlit UI (now that components are verified)
- Add more test queries to this notebook for regression testing
"""))